In [ ]:
import pandas as pd
import requests
import json
import time
import nltk
import spacy

from nltk.corpus import framenet as fn
from nltk.corpus.reader.framenet import PrettyList
from nltk.tokenize import sent_tokenize
from openai import OpenAI
from tqdm import tqdm

In [ ]:
frames = list()
for frame in PrettyList(fn.frames()):
    frame_str = str()
    frame_str += f"{frame.name}"
    # frame_str += f"Frame:\n{frame.name}"
    # frame_str += f"\nDefinition:\n{frame.definition}"
    # frame_str += f"\nFrame Elements:\n"
    # num_fes = 1
    # for fe in frame.FE.keys():
    #     frame_str += f"{num_fes}. {fe}\n"
    #     num_fes += 1
    frame_str += '\n'
    frames.append(frame_str)

len(frames)

In [ ]:
for frame in PrettyList(fn.frames()):
    print(frame.frameRelations)
    break

In [ ]:
print(''.join(frames))

In [ ]:
reddit_df = pd.read_csv("../../corpora/en_reddit_threaded/processed_reddit.csv", header=None)
reddit = reddit_df[0].tolist()

reddit = reddit[:4]

In [ ]:
cnn_news_df = pd.read_csv("../../corpora/en_cnn_news/processed_cnn_news.csv", header=None)
cnn_news = cnn_news_df[0].tolist()

cnn_news = cnn_news[:4]

In [ ]:
with open("../../corpora/en_ukwac/processed_ukwac.txt", 'r') as f:
    ukwac = f.readlines()

ukwac = ukwac[:2]

In [ ]:
ukwac = reddit + cnn_news + ukwac

len(ukwac)

In [ ]:
tokenizer = nltk.data.load('tokenizers/punkt/english.pickle')

In [ ]:
nlp = spacy.load("en_core_web_sm")

In [ ]:
def split_into_clauses_spacy(sentence):
    """Splits a sentence into clauses using spaCy's dependency parser."""
    doc = nlp(sentence)
    clauses = []
    start = 0
    for i, token in enumerate(doc):
        # Detect clause boundaries based on syntactic relations (e.g., conjunctions, subordinating conjunctions)
        if token.dep_ == "cc" or token.dep_ == "mark": # cc: coordinating conjunction, mark: marker (subordinating conjunction)
            clauses.append(doc[start:token.i].text)
            start = token.i
    clauses.append(doc[start:].text)
    return [clause.strip() for clause in clauses if clause.strip()]

In [ ]:
example = """For the text: "She runs fast."

Words and annotations might be:

[
  {"word": "She", "fee": false, "frame": "-", "partof": "-"},
  {"word": "ran", "fee": true, "frame": "Fleeing", "partof": "ran away"},
  {"word": "away", "fee": false, "frame": "-", "partof": "ran away"},
  {"word": "fast", "fee": true, "frame": "Speed_description", "partof": "-"}
]
"""

In [ ]:
PROMPT = """TASK: Annotate the given TEXT word by word with Berkeley FrameNet frames from the provided FRAMES list.

INSTRUCTIONS:
1. Using Fillmore's Frame Semantics theory, inspect each word in the TEXT considering its context.
2. For every word, assign the parameters in JSON array format: 'word', 'fee', 'frame', and 'partof'.
3. Set 'fee' to TRUE if the word evokes or triggers a FrameNet frame (e.g., verbs denoting actions, nouns denoting entities or concepts related to frames). Otherwise, set 'fee' to FALSE.
4. If 'fee' is TRUE, then for the parameter 'frame' either select an appropriate frame from FRAMES or set it to 'To_Review', which means there are no suitable frames in FRAMES. You can only use frames from FRAMES list.
5. If the word is an adverb, then find the existing FrameNet frame it evokes and assign it. If there are no appropriate frames in FRAMES list, set 'frame' to 'To_Review'.
6. If you encounter a phrasal verb that evokes a FrameNet frame, then set 'frame' to the same FrameNet frame for each word in it. Additionally, put the whole phrasal verb as plain text in 'partof' for each word. If the current word is not a part of a phrasal verb, set 'partof' to '-'.
7. If the word is a proper name, then set 'fee' to FALSE.
8. If the word is an auxiliary verb (e.g. will, was, were, am, are, is), then set 'fee' to FALSE.
9. If 'fee' is FALSE, set 'frame' to '-'.
10. Provide a single JSON array object with all 'word', 'fee', 'frame', and 'partof' keys possessing their respective values.
11. Output only the JSON array object, no explanations or comments.
12. Do not revise and do not generate multiple answers.

EXAMPLE:
{example}

TEXT: {text}

FRAMES: {bfn_frames}
"""

In [ ]:
client = OpenAI(
  api_key=""
)

In [ ]:
responses = list()
all_sentences = list()
for text in tqdm(ukwac, position=0):
    sentences = tokenizer.tokenize(text)
    for sent in tqdm(sentences, position=1):
        subsentences = split_into_clauses_spacy(sent)
        for subsent in subsentences:
            all_sentences.append(subsent)

            text_prompt = PROMPT.format(example=example,
                                        text=subsent,
                                        bfn_frames=''.join(frames))
            completion = client.chat.completions.create(
                model="gpt-4.1-mini-2025-04-14",
                store=True,
                messages=[
                    {
                        "role": "user",
                        "content": text_prompt
                    }
                ]
            )

            response = completion.choices[0].message.content
            responses.append(response)
            
            # break

In [ ]:
print(responses[0])

In [ ]:
len(all_sentences)

In [ ]:
len(responses)

In [ ]:
df_dict = {"sents": all_sentences, "preds": responses}
df = pd.DataFrame(df_dict)

df

In [ ]:
df.to_csv("../../outputs/fee_annotation/en_10_gpt_preds_subsentences_adverbs.csv", index=False)